# Results Loader and Plots Comparison for all Models

In [1]:
from pathlib import Path

import pandas as pd

from config.config import Config
from src.data import time_series_split
from src.plots import plot_forecast_diagnostics, plot_forecast_overlay, plot_return_overlay, plot_test_overlay, plot_val_overlay
from src.utils import load_results_from_dir, results_to_df, set_seed

In [2]:
cfg = Config.load()
SEED = cfg.runtime.seed
HORIZON = cfg.runtime.horizon
TARGET_MODE = cfg.runtime.target_mode
SAVE = True
rng = set_seed(SEED)

OUT_DIR = Path(cfg.data.processed_dir)
FIG_DIR = Path(cfg.data.fig_dir)

2025-09-26 15:55:32,460 - INFO - src.utils - Global random seed set to 42


In [3]:
results = load_results_from_dir(OUT_DIR)
len(results), [r["kind"] for r in results]

(2, ['linreg', 'linreg_wo_sent'])

In [4]:
# keep only cnn and xgboost
keep = {"linreg", "linreg_wo_sent"}
results = [r for r in results if r["kind"] in keep]

In [5]:
df_full = pd.read_csv(Path(cfg.data.processed_dir) / "features_full.csv")
train, val, test, forecast = time_series_split(df_full, train_ratio=0.8, val_ratio=0.1, horizon=HORIZON)

In [6]:
params_summary = results_to_df(results, "best_params")
params_summary

,model,alpha,l1_ratio,penalty,max_iter,learning_rate,eta0,random_state
0,linreg,0.01,0.832443,l1,2000,constant,0.01,42
1,linreg_wo_sent,0.01,0.832443,l1,2000,constant,0.01,42


In [7]:
metrics_summary = results_to_df(results, ["metrics", "test", "aggregate"])
metrics_summary

,model,mae,mse,rmse,r2,directional_accuracy
0,linreg,0.021741,0.000960,0.030979,-0.054027,0.476764
1,linreg_wo_sent,0.021119,0.000937,0.030609,-0.029028,0.507745


In [10]:
metrics_summary = results_to_df(results, ["metrics", "test", "per_horizon"])
horizon_list = cfg.runtime.horizon_list
mapping = {i+1: h for i, h in enumerate(horizon_list)}
metrics_summary["per_horizon"] = metrics_summary["per_horizon"].astype(int).map(mapping)
metrics_summary = metrics_summary.rename(columns={"per_horizon": "horizon_days"})
metrics_summary = metrics_summary.sort_values(
    by=["horizon_days", "model"],
    ascending=[True, True]
).reset_index(drop=True)
metrics_summary

,model,horizon_days,target_std,mae,mse,rmse,r2,directional_accuracy
0,linreg,1,0.010940,0.008357,0.000125,0.011171,-0.048052,0.523316
1,linreg_wo_sent,1,0.010940,0.008188,0.000120,0.010943,-0.005626,0.518135
2,linreg,5,0.023799,0.018787,0.000602,0.024527,-0.067625,0.432990
3,linreg_wo_sent,5,0.023799,0.017945,0.000573,0.023947,-0.017681,0.438144
4,linreg,20,0.045318,0.038078,0.002153,0.046397,-0.053632,0.474227
5,linreg_wo_sent,20,0.045318,0.037226,0.002118,0.046017,-0.036458,0.567010
